In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import psycopg2
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import sklearn
import tensorflow as tf

# Import necessary global variables and modules
from config.config import *
import assets.helpers as hp
import assets.metrics as m
import assets.data_splitting as ds

# Load data

## From SQL:

In [ ]:
# Connect to db
conn = psycopg2.connect(dbname=DBNAME, user='postgres', password='postgres')
cur = conn.cursor() 

# Read vital signs
vitals = pd.read_sql_query(f'SELECT * FROM vitals_windowed_{WINDOW_LENGTH:d}h_min{MIN_LOS_ICU:d}h;', conn)
# Read in labs values
labs = pd.read_sql_query(f'SELECT * FROM labs_windowed_{WINDOW_LENGTH:d}h_min{MIN_LOS_ICU:d}h;', conn)
# Read in notes mortality risk values
notes = pd.read_sql_query(f'SELECT * FROM notes_windowed_{WINDOW_LENGTH:d}h_min{MIN_LOS_ICU:d}h_{TEXT_ENCODER}{ABLATION_TYPE};', conn)

# Close the cursor and connection to so the server can allocate bandwidth to other requests
cur.close()
conn.close()

In [ ]:
if ADD_VITALS:    
    print("########## vitals info: ##########\n")
    print(vitals.info())
    print()
    print("########## vitals head: ##########\n")
    print(vitals.head(3))

In [ ]:
if ADD_LABS:
    print("########## labs info: ##########\n")
    print(labs.info())
    print()
    print("########## labs head: ##########\n")
    print(labs.head(3))

In [ ]:
if ADD_NOTES:
    print("########## notes info: ##########\n")
    print(notes.info())
    print()
    print("########## notes head: ##########\n")
    print(notes.head(3))

In [ ]:
# Set data and obtain 'icustays' indeces to later get the train and test splits
icustays, vitals_grouped, labs_grouped, notes_grouped = ds.set_data(vitals, labs, notes, key='icustay_id', label='label_death_icu')

In [ ]:
# Load train and test indeces

i_train = pd.read_csv(I_TRAIN_PATH)
i_train = i_train.to_numpy().flatten()

i_test = pd.read_csv(I_TEST_PATH)
i_test = i_test.to_numpy().flatten()

In [ ]:
print("Train:")
print(i_train)
print(f"The train set has {len(i_train)} ICU stays.\n")

print("Test:")
print(i_test)
print(f"The test set has {len(i_test)} ICU stays.")

In [ ]:
# Use the data coming from the i_train/i_test splits, for later setting the trainer

# Convert icustays into a DataFrame
icustays = pd.DataFrame(icustays)

# Assuming the 'icustay_id' column is the first column (index 0)
# Get the 'icustay_id' values for the training and testing splits
icustay_ids_train = icustays.iloc[i_train, 0]
icustay_ids_test = icustays.iloc[i_test, 0]

# Split 'vitals' DataFrame
vitals_train = vitals[vitals['icustay_id'].isin(icustay_ids_train)]
vitals_test = vitals[vitals['icustay_id'].isin(icustay_ids_test)]
vitals = pd.concat([vitals_train, vitals_test], ignore_index=True)

# Split 'labs' DataFrame
labs_train = labs[labs['icustay_id'].isin(icustay_ids_train)]
labs_test = labs[labs['icustay_id'].isin(icustay_ids_test)]
labs = pd.concat([labs_train, labs_test], ignore_index=True)

# Split 'notes' DataFrame
notes_train = notes[notes['icustay_id'].isin(icustay_ids_train)]
notes_test = notes[notes['icustay_id'].isin(icustay_ids_test)]
notes = pd.concat([notes_train, notes_test], ignore_index=True)

print('Number of unique ICU stays in vitals:', len(vitals['icustay_id'].unique()))
print('Number of unique ICU stays in lab:', len(labs['icustay_id'].unique()))
print('Number of unique ICU stays in notes:', len(notes['icustay_id'].unique()))

# Data Processing
Create interface specifications: 

In [ ]:
specs = []
if ADD_VITALS:
    vitals_spec = tf.TensorSpec(shape=(None, len(VITAL_NAMES)), dtype=tf.dtypes.float64, name='vitals')
    specs.append(vitals_spec)
if ADD_LABS:
    labs_spec = tf.TensorSpec(shape=(None, len(LAB_NAMES)), dtype=tf.dtypes.float64, name='labs')
    specs.append(labs_spec)
if ADD_NOTES:
    notes_spec = tf.TensorSpec(shape=(None, len(NOTES_NAMES)), dtype=tf.dtypes.float64, name='notes')
    specs.append(notes_spec)

label_spec = tf.TensorSpec(shape=1, dtype=tf.dtypes.float64, name='label')

output_signature = (tuple(specs), label_spec)

## Build Model 

Build Model:

In [ ]:
branches = []
inputs = []

# Vital signs stream
if ADD_VITALS:
    inputs_vitals = tf.keras.Input(shape=vitals_spec.shape, name='Input_vitals')
    mask_vitals = tf.keras.layers.Masking(mask_value=-2., name='mask_vitals')(inputs_vitals)
    stack_output_vitals = hp.create_model_stack(mask_vitals, MODEL_ARCHITECTURE, 'vitals')
    normalized_vitals = tf.keras.layers.BatchNormalization(name='BatchNorm_vitals')(stack_output_vitals)
    inputs.append(inputs_vitals)
    branches.append(normalized_vitals)

# Lab results stream
if ADD_LABS:
    inputs_labs = tf.keras.Input(shape=labs_spec.shape, name='Input_labs')
    mask_labs = tf.keras.layers.Masking(mask_value=-2., name='mask_labs')(inputs_labs)
    stack_output_labs = hp.create_model_stack(mask_labs, MODEL_ARCHITECTURE, 'labs')
    normalized_labs = tf.keras.layers.BatchNormalization(name='BatchNorm_labs')(stack_output_labs)
    inputs.append(inputs_labs)
    branches.append(normalized_labs)

# Notes mortality scores stream
if ADD_NOTES:
    inputs_notes = tf.keras.Input(shape=notes_spec.shape, name='Input_notes')
    mask_notes = tf.keras.layers.Masking(mask_value=-2., name='mask_notes')(inputs_notes)
    stack_output_notes = hp.create_model_stack(mask_notes, MODEL_ARCHITECTURE, 'notes')
    normalized_notes = tf.keras.layers.BatchNormalization(name='BatchNorm_notes')(stack_output_notes)
    inputs.append(inputs_notes)
    branches.append(normalized_notes)

# Concatanation of branches
if len(branches) > 1:
    merge = tf.keras.layers.Concatenate()(branches)
else:
    merge = branches[0]

FCL1 = tf.keras.layers.Dense(16, name='FCL1')(merge)
FCL2 = tf.keras.layers.Dense(16, name='FCL2')(FCL1)
outputs = tf.keras.layers.Dense(1, activation='sigmoid', name='output')(FCL2)

model = tf.keras.Model(inputs=inputs, outputs=outputs, name=f'{MODEL_ARCHITECTURE}_model')
model.summary()

## Evaluate Model:

In [ ]:
# Loss Function
loss_fcn = 'binary_crossentropy'

# Metrics
metrics=[
    m.ContinuousAUC(curve='ROC', name='AUROC'),
    m.ContinuousAUC(curve='PR', name='AUPRC'),
    m.ContinuousF1(name='F1'),
    m.ContinuousRecall(name='recall'),
    m.ContinuousPrecision(name='precision')
]

In [ ]:
# Create Trainer object

if USE_FL:
    trainer_class = hp.TrainerFL
else:
    trainer_class = hp.Trainer

trainer = trainer_class(
    loss_fcn,
    metrics,
    output_signature=output_signature,
    n_clients=CLIENT_COUNT,
    es_metric='F1',
    es_mode='max',
    min_los_icu=MIN_LOS_ICU,
    n_folds=1,
    max_threads=2,
    random_state=RANDOM_STATE,
    i_train=i_train,
    i_test=i_test
)

In [ ]:
# Set data
trainer.set_data(
    vitals[['icustay_id', 'label_death_icu'] + (VITAL_NAMES if ADD_VITALS else [])],
    labs[['icustay_id', 'label_death_icu'] + (LAB_NAMES if ADD_LABS else [])] if ADD_LABS else None,
    notes[['icustay_id', 'label_death_icu'] + NOTES_NAMES] if ADD_NOTES else None
)

In [ ]:
trainer.evaluate(
    model,
    n_labels=2,
    weighted=True,
    shuffle=True,
    stratify_clients=True
)  

## Statistics

In [ ]:
print('Average test AUROC:', trainer.test_scores['AUROC'].mean())
print('Average test AUPRC:', trainer.test_scores['AUPRC'].mean())
print('Average test F1:', trainer.test_scores['F1'].mean())